# KBEL demo: `extract`, `link` and `construct`

This notebook shows the three main methods of `KBEL`:

| Method | Input | Output |
|--------|-------|--------|
| `link` | a `Mention` | the knowledge-base entities it refers to |
| `extract` | a text | `Triple(subject, predicate, object)` strings |
| `construct` | a document | KIF `Statement`s whose subject, property and value were all linked |

## Setup

In [3]:
import os
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
os.environ.setdefault('TRANSFORMERS_VERBOSITY', 'error')

import logging
logging.basicConfig(level=logging.ERROR)

try:                      
    import dotenv
    dotenv.load_dotenv()
except ImportError:
    pass

from kbel.core.kbel import KBEL
from kbel.core.mention import Mention, EntityType

# Plugins register themselves on import.
from kbel.disambiguators import naive, similarity, llm 
from kbel.extractors import mrebel, openie

In [4]:
def entity(result):
    """`label (id)` of a LinkResult."""
    return f"{result.candidate.label} ({result.candidate.iri.rsplit('/', 1)[-1]})"

## 1. `link`: from a mention to an entity

A `Mention` has the `term` to link, the sentence it appears in (`text`) and
whether it is an item or a property. `link` searches the knowledge base for
candidates and lets the **disambiguator** choose.

In [21]:
kbel = KBEL('wikidata', disambiguator='naive')

mention = Mention(term='Python', text='Python is a programming language used for coding.')

for result in kbel.link(mention):
    print(entity(result), '-', result.candidate.description)

Python (Q28865) - general-purpose programming language


The `naive` disambiguator ignores the sentence: it always picks the candidate
whose label matches the term. The `sim` disambiguator compares the sentence
with each candidate's description, so the same term can link to different
entities:

In [34]:
snake = Mention(term='Python', text='The python is a large snake that kills its prey by constriction.')

for name in ('naive', 'sim'):
    kbel.disambiguator = name
    print(f'{name:5s}  programming -> {entity(kbel.link(mention)[0]):22s}  snake -> {entity(kbel.link(snake)[0])}')

naive  programming -> Python (Q28865)         snake -> Python (Q28865)
sim    programming -> Python (Q28865)         snake -> python (Q184018)


`sim` also returns a ranked list with scores; `limit` sets how many results:

In [35]:
for result in kbel.link(snake, limit=3):
    print(f'{result.score:.3f}  {entity(result):30s} {result.candidate.description}')

0.684  python (Q184018)               family of snakes
0.612  Burmese python (Q2716137)      species of large nonvenomous snake
0.610  Python (Q271218)               genus of reptiles


### Linking properties

With `entity_type=EntityType.PROPERTY` the search looks for properties
(relations) instead of items:

In [36]:
for term in ('place of birth', 'born in', 'created by'):
    [result] = kbel.link(Mention(term=term, text='', entity_type=EntityType.PROPERTY), limit=1)
    print(f'{term:15s} -> {entity(result)}')

place of birth  -> place of birth (P19)
born in         -> place of birth (P19)
created by      -> creator (P170)


The knowledge base is also chosen by name — e.g. `wikidata-vdb`, a semantic
(vector) search over Wikidata:

In [37]:
kbel.knowledge_base = 'wikidata-vdb'
print(entity(kbel.link(mention, limit=1)[0]))

Python (Q28865)


## 2. `extract`: from text to triples

An **extractor** turns text into triples. The two used here work differently:

- `mrebel` is a neural model trained on Wikidata: predicates are Wikidata
  property labels (`place of birth`), and it supports 18 languages.
- `openie` is Stanford OpenIE (rule based, requires Java): predicates are
  phrases from the text (`was born in`).

In [38]:
TEXT = ('Marie Curie was a Polish and naturalised-French physicist and chemist who was born in Warsaw. '
        'Marie Curie won the Nobel Prize in Physics in 1903.')

kbel = KBEL('wikidata', extractor='mrebel')

for triple in kbel.extract(TEXT):
    print(triple)

Triple(subject='Marie Curie', predicate='place of birth', object='Warsaw')
Triple(subject='Marie Curie', predicate='occupation', object='physicist')
Triple(subject='Marie Curie', predicate='occupation', object='chemist')
Triple(subject='Marie Curie', predicate='country of citizenship', object='French')
Triple(subject='Marie Curie', predicate='award received', object='Nobel Prize in Physics')
Triple(subject='Nobel Prize in Physics', predicate='winner', object='Marie Curie')
Triple(subject='Marie Curie', predicate='floruit', object='1903')


In [39]:
kbel.extractor = 'openie'

for triple in kbel.extract(TEXT):
    print(triple)

Triple(subject='Marie Curie', predicate='was', object='Polish physicist')
Triple(subject='chemist', predicate='was born in', object='Warsaw')
Triple(subject='Marie Curie', predicate='was physicist', object='born in Warsaw')
Triple(subject='Marie Curie', predicate='was Polish physicist', object='born in Warsaw')
Triple(subject='Marie Curie', predicate='was Polish and naturalised-French physicist', object='born in Warsaw')
Triple(subject='Curie', predicate='was Polish', object='born')
Triple(subject='chemist', predicate='was', object='born')
Triple(subject='Marie Curie', predicate='won Nobel Prize in', object='1903')
Triple(subject='Marie Curie', predicate='won in', object='1903')
Triple(subject='Marie Curie', predicate='won Nobel Prize in', object='Physics')
Triple(subject='Marie Curie', predicate='won', object='Nobel Prize')
Triple(subject='Marie Curie', predicate='won in', object='Physics')


Extractors take their own options. For instance, `mrebel` can read other
languages — the predicates still come back as English Wikidata labels:

In [40]:
from kbel.extractors import TripleExtractor

kbel.extractor = TripleExtractor('mrebel', source_language='pt_XX')

for triple in kbel.extract('Ada Lovelace trabalhou com Charles Babbage na Máquina Analítica, em Londres.'):
    print(triple)

Triple(subject='Máquina Analítica', predicate='discoverer or inventor', object='Charles Babbage')
Triple(subject='Charles Babbage', predicate='notable work', object='Máquina Analítica')


## 3. `construct`: from a document to KIF statements

`construct` combines both: it splits the document into paragraphs and
sentences, extracts triples from each sentence, links subject (item),
predicate (property) and object (item) — using the sentence as the mention's
`text` and the paragraph as its `context` — and yields a
[KIF](https://github.com/PUC-Behring-AI/kif) **`Statement`** for every triple
whose three parts were linked:

```
Statement(subject, ValueSnak(property, value))
```

In [ ]:
%env KIF_DEBUG=
%env KIF_RESOLVE_ENTITIES=1
from kif_lib import *
from kif_lib.vocabulary import wd

DOCUMENT = """Marie Curie was a Polish and naturalised-French physicist and chemist. Marie Curie was born in Warsaw.

Pierre Curie was a French physicist. Pierre Curie won the Nobel Prize in Physics in 1903 with Marie Curie."""

kbel = KBEL('wikidata', disambiguator='naive', extractor='mrebel')

statements = kbel.construct(DOCUMENT, language='en')

display(*statements)

env: KIF_DEBUG=
env: KIF_RESOLVE_ENTITIES=1


(**Statement** (**Item** [Marie Curie](http://www.wikidata.org/entity/Q7186)) (**ValueSnak** (**Property** [occupation](http://www.wikidata.org/entity/P106)) (**Item** [wd:Q169470](http://www.wikidata.org/entity/Q169470))))

(**Statement** (**Item** [Marie Curie](http://www.wikidata.org/entity/Q7186)) (**ValueSnak** (**Property** [occupation](http://www.wikidata.org/entity/P106)) (**Item** [wd:Q593644](http://www.wikidata.org/entity/Q593644))))

(**Statement** (**Item** [Marie Curie](http://www.wikidata.org/entity/Q7186)) (**ValueSnak** (**Property** [country of citizenship](http://www.wikidata.org/entity/P27)) (**Item** [wd:Q150](http://www.wikidata.org/entity/Q150))))

(**Statement** (**Item** [Marie Curie](http://www.wikidata.org/entity/Q7186)) (**ValueSnak** (**Property** [country of citizenship](http://www.wikidata.org/entity/P27)) (**Item** [wd:Q809](http://www.wikidata.org/entity/Q809))))

(**Statement** (**Item** [Marie Curie](http://www.wikidata.org/entity/Q7186)) (**ValueSnak** (**Property** [place of birth](http://www.wikidata.org/entity/P19)) (**Item** [Warsaw](http://www.wikidata.org/entity/Q270))))

... (4 outputs left out)

(**Statement** (**Item** [wd:Q37463](http://www.wikidata.org/entity/Q37463)) (**ValueSnak** (**Property** [award received](http://www.wikidata.org/entity/P166)) (**Item** [wd:Q38104](http://www.wikidata.org/entity/Q38104))))

(**Statement** (**Item** [wd:Q37463](http://www.wikidata.org/entity/Q37463)) (**ValueSnak** (**Property** [spouse](http://www.wikidata.org/entity/P26)) (**Item** [Marie Curie](http://www.wikidata.org/entity/Q7186))))

(**Statement** (**Item** [Marie Curie](http://www.wikidata.org/entity/Q7186)) (**ValueSnak** (**Property** [spouse](http://www.wikidata.org/entity/P26)) (**Item** [wd:Q37463](http://www.wikidata.org/entity/Q37463))))

(**Statement** (**Item** [Marie Curie](http://www.wikidata.org/entity/Q7186)) (**ValueSnak** (**Property** [award received](http://www.wikidata.org/entity/P166)) (**Item** [wd:Q38104](http://www.wikidata.org/entity/Q38104))))

### Async

Extraction, searches and disambiguations run concurrently (at most
`max_concurrency` at a time). In async code — Jupyter included — iterate
the async version:

In [9]:
async for statement in kbel.aconstruct(DOCUMENT, language='en', max_concurrency=8):
    display(statements)

<generator object _iterate_sync at 0x10ec26b00>

<generator object _iterate_sync at 0x10ec26b00>

<generator object _iterate_sync at 0x10ec26b00>

<generator object _iterate_sync at 0x10ec26b00>

<generator object _iterate_sync at 0x10ec26b00>

<generator object _iterate_sync at 0x10ec26b00>

... (3 outputs left out)

<generator object _iterate_sync at 0x10ec26b00>

<generator object _iterate_sync at 0x10ec26b00>

<generator object _iterate_sync at 0x10ec26b00>

<generator object _iterate_sync at 0x10ec26b00>

`link` has an async version too, so several mentions can be linked at once:

In [ ]:
import asyncio

mentions = [Mention(term=t, text='') for t in ('Marie Curie', 'Warsaw', 'polonium', 'radium')]
results = await asyncio.gather(*(kbel.alink(m, limit=1) for m in mentions))

for m, [result] in zip(mentions, results):
    print(f'{m.term:12s} -> {entity(result)}')

Marie Curie  -> Marie Curie (Q7186)
Warsaw       -> Warsaw (Q270)
polonium     -> polonium (Q979)
radium       -> radium (Q1128)


Any combination of knowledge base, extractor and disambiguator works with
these methods — for example `KBEL('wikidata-vdb', disambiguator='sim',
extractor='openie')`.